<a href="https://colab.research.google.com/github/lapshinaaa/recsys-tasks/blob/main/DeepRecSys1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep RecSys Course

In this assignment, we will go through the full basic pipeline: data
preparation → metrics → several recommendation approaches → final
leaderboard.

### Libraries Used
In the following task we're going to be using the following libs:
* [polars](https://pola.rs/) - for data processing
* [implicit](https://github.com/benfred/implicit) - library for using and training collaborative recsys
* [torch](https://pytorch.org/) - no comments
* [gensim](https://radimrehurek.com/gensim/) - training **word2vec**

### Data
Data is in `data.zip`, which consists of:
* `interactions.parquet` - user-item interactions from Yambda dataset (likes for the 500m version)
* `embeddings.parquet` - pre-filtered and more densely packed track embeddings from Yambda
* `artists.parquet` - item metadata with mapping to artists

Archive can be downloaded from: [here](https://drive.google.com/file/d/1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS/view?usp=sharing).

### Guidelines
-   Determinism: fix random seeds where necessary
-   Do not use test data during training/model preparation
-   Test set = the last week (by timestamp), as described below
-   After each stage, run the checks inside the notebook
-   Try to avoid working with raw Python objects (dicts, lists, ints) where `polars` methods can be applied — they will be tens to hundreds of times faster and more readable
-   To quickly detect inefficient code that runs too long, wrap loops with `tqdm` and display a progress bar
-   During debugging, you may sample the data for speed using `data.sample(fraction=0.1, seed=42)`. After debugging, run tests on the full dataset


In [1]:
import tests

!pip install gensim
!pip install implicit

!pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -q dataset.zip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 79.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=10906356 sha256=4e75efb48df7912d345b81871e1e666e0849618ca7f1aab36248f537f0d23a6a
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&

### 1. Data Preparation

**Tasks:**
1) Load the data (interactions, embeddings, metadata).
2) Keep only interactions for which embeddings exist.
3) Join metadata (track artists) to all interactions.
4) Apply core filtering: keep only items with ≥5 interactions (in this
homework we do this purely for convenience and speed; in real work this
should not be done blindly)
5) Perform train-test split: put the last week into test.
6) Restrict the test set to users who have interactions in train.
7) For quality assessment, prepare `test_targets: Dict[uid, List[item_id]]`.

After this block we should have: `train`, `test`, `embeddings`, `artists`, `test_targets`.

For this block the following methods will be useful:
* `pl.read_parquet` - for reading data
* `.filter, .value_counts` will help perform core-filtration
* `df.join(...)` - when joining metadata `how='left'` must be used, not `how='inner'`
* `df.join(other, on=some_key, how='semi')` - the `semi` mode is used for filtration (keep only those rows from the original DataFrame
whose key exists in the second table)

In [1]:
from typing import Dict, List, Tuple, Any, Optional
from collections import defaultdict
import os
from tqdm.auto import tqdm
import numpy as np
import polars as pl
import math

import tests
import torch
import torch.nn.functional as F
import heapq

# data paths
DATA_DIR = "."
PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")
PATH_EMBEDDINGS = os.path.join(DATA_DIR, "embeddings.parquet")
PATH_ARTISTS = os.path.join(DATA_DIR, "artists.parquet")

# global variables
TOPK = 100
CORE_MIN_INTERACTIONS_PER_ITEM = 5
TEST_INTERVAL_SECONDS = 7 * 24 * 60 * 60

# for reproducibility
np.random.seed(42)

data = pl.read_parquet(PATH_INTERACTIONS)
embeddings = pl.read_parquet(PATH_EMBEDDINGS)
artists = pl.read_parquet(PATH_ARTISTS)

###########################
### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
###########################

2. Keeping only interactions for which embeddings exist

In [2]:
data = data.join(
    embeddings.select("item_id"),
    on="item_id",
    how="semi"
)

In [3]:
embeddings.head(2)

item_id,embed
u32,"array[f32, 128]"
26,"[1.795751, -0.417789, … 2.790108]"
50,"[-1.348751, 1.248032, … -1.081478]"


In [5]:
#data.head(2)

In [6]:
data['item_id'].unique().shape

(591811,)

In [7]:
embeddings.shape[0]

591811

In [8]:
data.shape[0]

8687987

3-4. Core filtering and joining metadata

In [4]:
popular_items = (
    data
    .select("item_id")
    .to_series()
    .value_counts()
    .filter(pl.col("count") >= CORE_MIN_INTERACTIONS_PER_ITEM)
    .select("item_id")
)

data = data.join(popular_items, on="item_id", how="semi")

In [5]:
data = data.join(
    artists,
    on="item_id",
    how="left"
)

In [11]:
#data.shape[0]

5-6. Train-test split and restriction

In [6]:
max_ts = data.select(pl.col("timestamp").max()).item()
split_ts = max_ts - TEST_INTERVAL_SECONDS

In [8]:
split_ts

25395195

In [7]:
# perform split
train = data.filter(pl.col("timestamp") < split_ts)
test  = data.filter(pl.col("timestamp") >= split_ts)

In [10]:
train.shape[0]

7755988

In [8]:
test = test.join(train.select("uid").unique(), on="uid", how="semi")

7. `test_targets` prep for quality assessment

In [9]:
test_targets_df = (
    test
    .group_by("uid")
    .agg(pl.col("item_id").alias("item_id"))
)

In [10]:
test_targets = dict(
    zip(
        test_targets_df["uid"].to_list(),
        test_targets_df["item_id"].to_list()
    )
)

In [14]:
len(test_targets)

37446

In [11]:
# tests py

tests.check_data_split(train=train, test=test, embeddings=embeddings, artists=artists, test_targets=test_targets)

All good! :)


### 2. Quality Assessment

#### 2.1 Defining metrics

2.1.1 Let for the user $u$:

* $G_u \subset \mathcal{I}$ — be the set of relevant items (ground truth)
* $R_u = (r_{u,1}, \dots, r_{u,K})$ — ordered list of recommendations of length $K$

Let's define the relevance indicator as $I_{u,k} = [ r_{u, k} \in G_u]$. Or it can be called `hits`.

2.1.2 **Hitrate@K** is equal to 1, if we guessed among top@k at least 1 relevant item:
* $
\text{Hitrate@K} = \frac{1}{|U|}
\sum_{u \in U}
\left[ \sum_{k=1}^{K} I_{u,k} > 0 \right]
$

2.1.3 **Recall@K** assess a portion of guessed relevant items (of all relevant items):
* $
\text{Recall@K} = \frac{1}{|U|}
\sum_{u \in U}
\frac{
\sum_{k=1}^{K} I_{u,k}
}{
\min(|G_u|, K)
}
$

2.1.4 For calculating **nDCG@K** we should first calculate **DCG@K**, then calculate **iDCG@K** (DCG in case of ideal ranging), then divide one by the other:
* $
\text{DCG@K}(u) = \sum_{k=1}^{K}
\frac{I_{u,k}}{\log_2(k+1)}
$
* $
\text{iDCG@K}(u) = \sum_{k=1}^{\min(|G_u|,K)}
\frac{1}{\log_2(k+1)}
$
* $
\text{nDCG@K} = \frac{1}{|U|}
\sum_{u \in U}
\frac{\text{DCG@K}(u)}{\text{iDCG@K}(u)}
$

2.1.5 **Coverage@K** - number of unique items in all recommendations, divided by the size of a catalog:

* $
\text{Coverage@K} = \frac{|\bigcup_{u \in U} R_u|}{|\mathcal{I}_{train}|},
$ where $\mathcal{I}_{train}$ — item catalog in train.
* as the size of a catalog we're going to be using number of items that are available for recommendation the moment of recommendation (essentially, the number of unique items in `train`)

#### 2.2 What needs to be done
Implement functions:
- `get_metrics(targets, candidates, topk) -> dict(hitrate, recall, ndcg)`
- `evaluate(targets_by_user, candidates_by_user, catalog_size, topk) -> dict(hitrate, recall, ndcg, coverage)`

**Important:** `candidates[uid]` must of length precisely `topk`.  


In [12]:
def get_metrics(targets: List[int], candidates: List[int], topk: int) -> Dict[str, float]:
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################

    """
    Per-user metrics. targets = relevant items G_u, candidates = ranked list R_u (length >= topk).
    Returns hitrate@k(u), recall@k(u), ndcg@k(u).
    """

    recs = candidates[:topk]
    gt = set(targets)

    hits = [1 if item in gt else 0 for item in recs] # faster since gt is a dict
    num_hits = sum(hits)

    # Hitrate@K(u)
    hitrate = 1.0 if num_hits > 0 else 0.0

    # Recall@K(u)
    denom = min(len(targets), topk) # for len can't use len, MUST use original targets
    recall = (num_hits / denom) if denom > 0 else 0.0

    # DCG@K(u)
    dcg = 0.0
    for idx, h in enumerate(hits, start=1):  # idx = 1..K
        if h:
            dcg += 1.0 / math.log2(idx + 1)

    # iDCG@K(u)
    idcg = 0.0
    for idx in range(1, denom + 1):
        idcg += 1.0 / math.log2(idx + 1)

    ndcg = (dcg / idcg) if idcg > 0 else 0.0

    return {"hitrate": hitrate, "recall": recall, "ndcg": ndcg}


def evaluate(
    targets: Dict[int, List[int]],
    candidates: Dict[int, List[int]],
    catalog_size: int,
    topk: int = 100,
) -> Dict[str, float]:
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################
    """
    Aggregates metrics across users and computes coverage@K.
    Assumes candidates[uid] has length at least topk (or exactly topk, as your note says).
    """
    uids = list(targets.keys())

    hitrate_sum = 0.0
    recall_sum = 0.0
    ndcg_sum = 0.0

    # coverage: union of all recommended items across users in topK
    covered_items = set()

    for uid in uids:
        gt_u = targets[uid]
        rec_u = candidates[uid][:topk]  # guarantee topk slice

        m = get_metrics(gt_u, rec_u, topk=topk)
        hitrate_sum += m["hitrate"]
        recall_sum += m["recall"]
        ndcg_sum += m["ndcg"]

        covered_items.update(rec_u)

    num_users = len(uids)
    hitrate = hitrate_sum / num_users if num_users > 0 else 0.0
    recall = recall_sum / num_users if num_users > 0 else 0.0
    ndcg = ndcg_sum / num_users if num_users > 0 else 0.0

    coverage = (len(covered_items) / catalog_size) if catalog_size > 0 else 0.0

    return {"hitrate": hitrate, "recall": recall, "ndcg": ndcg, "coverage": coverage}

In [22]:
tests.check_metrics(get_metrics=get_metrics, evaluate=evaluate)

All good! :)


### 3. Popular Top

Create top popular items based on train interactions and calculate metrics using `evaluate`.

Helpful methods from `polars`: `.value_counts, .sort, .head, .to_numpy, .tolist, .n_unique`.

**Important:** in this task, you don't need to filter items for each user that have already been in their history. You just need the same list of candidates for all users.

In [13]:
###########################
### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
###########################

topk = TOPK

top_items = (
    train
    .select("item_id")
    .to_series()
    .value_counts()
    .sort("count", descending=True)
    .head(topk)
    .get_column("item_id")
    .to_list()
)


In [24]:
candidates_toppop = {uid: top_items for uid in test_targets.keys()} # essentialy, same recs for all users

catalog_size = train.select(pl.col("item_id").n_unique()).item()

In [25]:
metrics_toppop = evaluate(
    targets=test_targets,
    candidates=candidates_toppop,
    catalog_size=catalog_size,
    topk=topk
)

In [26]:
metrics_toppop

{'hitrate': 0.11237515355445174,
 'recall': 0.030365888143506912,
 'ndcg': 0.010997994080361362,
 'coverage': 0.0006363063687904452}

In [27]:
# put the results of evaluate into dict metrics_toppop,
# later on, in the notebook, correct names of dicts with metrics are expected
tests.check_top_pop(metrics_toppop)

All good! :)


**Important!** Tests above require exact match between reference notebook and current experiments to make sure data is processed correctly.
Later on, only surpassing a certain threshold will be required, not a precise match of values. Thresholds are much smaller than the ones from reference notebook.

### 4. Artist Recs

What we want to do here:
 1. Take the last N likes of the user
(for models and all calculations we must use ONLY the train set; the test set is used exclusively for evaluation).
 2. Based on these interactions, determine the user’s favorite artists
(sort artists by the number of likes the user gave to each artist).
 3. For those favorite artists, take their most popular tracks
(to compute track popularity, use only the train set).
 4. Keep only those tracks that the user has not seen (has not liked) before.
 5. Recommend those tracks.


In essence, we construct recommendations using simple counting statistics:
 * how many times the user liked an artist
 * how many times a track by that artist was liked overall


With this method, not every user will end up with 100 candidates, so we need to extend the recommendation list to 100 using a fallback strategy.

In this case the suggested fallback is `top_pop` recommendations:
 * if the user has fewer than 100 recommendations, iterate through the top_pop list and add candidates until the list reaches 100 items
 * while doing this, only add tracks that were not already included in the recommendations generated before the fallback


In [28]:
# train.head(5)

In [14]:
# values of global vars for tests.check_artist_recs
N_LAST_EVENTS = 100  # n most recent interactions
PER_ARTIST_LIMIT = 20  # for recs taking 20 items for each artist


# this func will be reused in the following tasks
def fallback_to_toppop(cands_by_uid: Dict[int, List[int]], top_pop: np.ndarray, topk: int) -> Dict[int, List[int]]:
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################
    result = {}

    for uid, cands in cands_by_uid.items():
        filled = list(cands)
        used = set(filled)

        for item in top_pop:
            if len(filled) >= topk:
                break
            if int(item) not in used:
                filled.append(int(item))
                used.add(int(item))

        result[uid] = filled

    return result


###########################
### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
###########################
last_user_events = (
    train
    .sort(["uid", "timestamp"], descending=[False, True])
    .group_by("uid")
    .head(N_LAST_EVENTS)
)

favorite_artists = (
    last_user_events
    .group_by(["uid", "artist_id"])
    .len()
    .rename({"len": "artist_likes"})
    .sort(["uid", "artist_likes"], descending=[False, True])
)

seen_by_user = (
    train
    .group_by("uid")
    .agg(pl.col("item_id").alias("seen_items"))
)

seen_dict = dict(zip(seen_by_user["uid"].to_list(), seen_by_user["seen_items"].to_list()))

In [30]:
fav_artists_df = favorite_artists.group_by("uid").agg(pl.col("artist_id").alias("artist_ids"))
fav_artists_dict = dict(zip(fav_artists_df["uid"].to_list(), fav_artists_df["artist_ids"].to_list()))

# top tracks per artist
artist_track_popularity = (
    train
    .group_by(["artist_id", "item_id"])
    .len()
    .rename({"len": "track_likes"})
    .sort(["artist_id", "track_likes"], descending=[False, True])
)

artist_top_tracks = ( # globally taking artists' top tracks
    artist_track_popularity
    .group_by("artist_id")
    .head(PER_ARTIST_LIMIT)
)

artist_tracks_df = artist_top_tracks.group_by("artist_id").agg(pl.col("item_id").alias("top_tracks"))
artist_tracks_dict = dict(zip(artist_tracks_df["artist_id"].to_list(), artist_tracks_df["top_tracks"].to_list()))

In [31]:
# build artist recs
artist_recs = {}

for uid in test_targets.keys():
    seen = set(seen_dict.get(uid, []))
    fav_artists = fav_artists_dict.get(uid, [])

    recs = []
    used = set()

    for artist_id in fav_artists:
        for item_id in artist_tracks_dict.get(artist_id, []):
            if item_id in seen:
                continue
            if item_id in used:
                continue

            recs.append(int(item_id))
            used.add(int(item_id))

            if len(recs) >= TOPK:
                break

        if len(recs) >= TOPK:
            break

    artist_recs[uid] = recs

In [32]:
artist_recs = fallback_to_toppop(
    cands_by_uid=artist_recs,
    top_pop=np.array(top_items),
    topk=TOPK
)

In [33]:
metrics_artist_recs = evaluate(
    targets=test_targets,
    candidates=artist_recs,
    catalog_size=catalog_size,
    topk=TOPK
)

In [34]:
metrics_artist_recs

{'hitrate': 0.21051647706030016,
 'recall': 0.05739755812576181,
 'ndcg': 0.022340091396709776,
 'coverage': 0.5952836971945252}

In [35]:
tests.check_artist_recs(metrics_artist_recs)

All good! :)


### 5. Item-to-Item recommendations

Here you're offered to implement a classic item-to-item algorithm:
1. For each item, calculating a list of similar items
2. Take N last user interactions, for each of them retrieving a list of similar ones
3. Aggregate candidates from the list of similar ones, taking into account cumulative similarities (if an item was seen in several lists of candidates for a user, sum similarities from each list)
4. Filter out everything that has already been seen
5. Keep only topk cands

The implementation must contain 2 functions:
1) `get_similar_items_gpu(item_ids, item_embeddings, topk_sim)` - for each item finding top similar items based on **cosine**
* using pytorch (and GPU), otherwise it'll take too long
* create tensor with embeddings, sending to GPU, applying $l_2$ regularization to embeddings (`torch.nn.functional.normalize`)
* batching over embeddings (going over embeddings in batches), for each batch using `torch.topk` calculating an honest top of similar; batches are important! Batching really does affect the speed of calculations.
* keeping this information in dict: `item_id -> [(cand_item_id, similarity), ...]`

2) `get_candidates_item2item(interactions, similar_items, ...)` - for each user agregating similarities based on N last interactions.
* we need to basically implement the aforementioned algorithm "taking last 10 interactions, sum similarities of all similar items"
* for the implementation of this func it's better not to try and implement the most optimal code using `polars` (it uses too much of CPU); it's better to use dicts, lists, etc. here
* you're offered to use `heapq.nlargest` to accelerate the search of `topk` elements among already calculated cumulative similarities (1.5-2 times faster than doing the full sort of all candidates from all lists)

In the second part it's important to also use **time decay**:
* fresh interactions must be of more importance, therefore when summing similaritites for a particular candidate, we need to consider the "freshness" of an event from the list of which we're taking the value of similarity.
* weight of an event $2^{-\frac{L-t}{\tau}}$, where $L$ - len of user history, $t$ - position of an event in history (starting from 1), $\tau$ - half-life, meaning how fast the signal from an event dies down. For example, when $\tau = 10$ the weight of an event loses twice its value if it comes 10 positions earlier in history.
* So we get $\text{score}(u, j) =
\sum_{t=1}^{L}
2^{- \frac{L - t}{\tau}}
\cdot
s(i_{u,t}, j)
$

**Important:** when using method `get_similar_items_gpu` we're calculating not 100 candidates, but twice as much (2 * topk = 200), so that after filtering and merging of lists we'll have more chances to get the required `topk`.

Helpful methods from `polars`: `.to_torch()`, `.to_numpy()` - they allow to immediately get tensors for ids/embeddings.

In [15]:
from tqdm.auto import tqdm

In [16]:
def get_similar_items_gpu(
    item_ids: np.ndarray,
    item_embeddings: np.ndarray,
    block: int = 32,
    topk: int = 200,
    device: str = "cuda",
    cand_block: int = 4096,
) -> Dict[int, List[Tuple[int, float]]]:
    """
    Returns dict item_id -> [(cand_item_id, similarity), ...] of len topk.
    Memory-safe exact top-k cosine similarity using chunking over both query items and candidate items.
    """
    item_ids = np.asarray(item_ids)
    item_embeddings = np.asarray(item_embeddings, dtype=np.float32)

    # keep normalized embeddings on CPU, move chunks to device when needed
    emb_cpu = torch.tensor(item_embeddings, dtype=torch.float32)
    emb_cpu = F.normalize(emb_cpu, p=2, dim=1)

    n_items = emb_cpu.shape[0]
    real_topk = min(topk, n_items - 1)
    result = {}

    for start in tqdm(range(0, n_items, block), desc="Query blocks"):
        end = min(start + block, n_items)
        q = emb_cpu[start:end].to(device)   # (B, D)
        bsz = end - start

        # running best candidates for this query block
        best_vals = torch.full((bsz, real_topk), -float("inf"), device=device)
        best_idxs = torch.full((bsz, real_topk), -1, dtype=torch.long, device=device)

        # go through all candidate items in chunks
        for c_start in range(0, n_items, cand_block):
            c_end = min(c_start + cand_block, n_items)
            c = emb_cpu[c_start:c_end].to(device)   # (C, D)

            sims = q @ c.T   # (B, C)

            # remove self-similarity where query block overlaps candidate chunk
            overlap_start = max(start, c_start)
            overlap_end = min(end, c_end)
            if overlap_start < overlap_end:
                rows = torch.arange(overlap_start - start, overlap_end - start, device=device)
                cols = torch.arange(overlap_start - c_start, overlap_end - c_start, device=device)
                sims[rows, cols] = -float("inf")

            chunk_k = min(real_topk, c_end - c_start)
            chunk_vals, chunk_local_idxs = torch.topk(sims, k=chunk_k, dim=1)
            chunk_global_idxs = chunk_local_idxs + c_start

            # merge chunk top-k with running top-k
            merged_vals = torch.cat([best_vals, chunk_vals], dim=1)
            merged_idxs = torch.cat([best_idxs, chunk_global_idxs], dim=1)

            new_vals, new_pos = torch.topk(merged_vals, k=real_topk, dim=1)
            new_idxs = torch.gather(merged_idxs, 1, new_pos)

            best_vals = new_vals
            best_idxs = new_idxs

            del c, sims, chunk_vals, chunk_local_idxs, chunk_global_idxs
            del merged_vals, merged_idxs, new_vals, new_pos, new_idxs
            if device == "cuda":
                torch.cuda.empty_cache()

        best_vals_np = best_vals.cpu().numpy()
        best_idxs_np = best_idxs.cpu().numpy()

        for i in range(bsz):
            src_item_id = int(item_ids[start + i])
            cand_item_ids = item_ids[best_idxs_np[i]]
            cand_sims = best_vals_np[i]

            result[src_item_id] = [
                (int(cand_id), float(sim))
                for cand_id, sim in zip(cand_item_ids, cand_sims)
                if cand_id != -1
            ]

        del q, best_vals, best_idxs, best_vals_np, best_idxs_np
        if device == "cuda":
            torch.cuda.empty_cache()

    return result


def get_candidates_item2item(
    interactions: pl.DataFrame,
    similar_items: Dict[int, List[Tuple[int, float]]],
    n_last: int = 30,
    half_life_frac: float = 0.5,
    topk: int = 100,
) -> Dict[int, List[int]]:
    """
    half_life_frac is interpreted as fraction of n_last: half_life = half_life_frac * n_last.
    Meaning, in the real formula you need to multiply the parameter half_life_frac by n_last
    w(r) = 2^{-r / half_life}, where r=0 is for the latest event.
    """
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################
    half_life = max(half_life_frac * n_last, 1e-8)

    # sorting interactions to later establish freshness
    hist_df = (
        interactions
        .sort(["uid", "timestamp"], descending=[False, True])
        .group_by("uid")
        .agg(pl.col("item_id").alias("history"))
    )
    history_by_user = dict(zip(hist_df["uid"].to_list(), hist_df["history"].to_list()))

    result = {}

    for uid, full_history in history_by_user.items():
      seen = set(full_history)
      fresh_history = full_history[:n_last]
      scores = defaultdict(float) # to accumulate scores

      for r, src_item in enumerate(fresh_history):
        weight = 2 ** (-r / half_life)

        for cand_item, sim in similar_items.get(src_item, []):
          if cand_item in seen:
            continue
          scores[cand_item] += weight * sim

      # only keeping topk
      top_cands = heapq.nlargest(topk, scores.items(), key=lambda x: x[1])
      result[uid] = [item_id for item_id, _ in top_cands]

    return result

In [17]:
device="cuda" if torch.cuda.is_available() else "cpu"

In [27]:
device

'cuda'

In [21]:
item_ids = embeddings["item_id"].to_numpy()
item_embeddings = embeddings["embed"].to_list()
item_embeddings = np.array(item_embeddings, dtype=np.float32)

In [24]:
item_embeddings.shape[0]

591811

In [ ]:
similar_items = get_similar_items_gpu(
    item_ids=item_ids,
    item_embeddings=item_embeddings,
    block=16,
    cand_block=2048,
    topk=2 * TOPK,
    device="cuda",
)

In [ ]:
###########################
### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
###########################
i2i_recs = get_candidates_item2item(
    interactions=train,
    similar_items=similar_items,
    n_last=30,
    half_life_frac=0.5,
    topk=TOPK,
)

metrics_i2i = evaluate(
    targets=test_targets,
    candidates=i2i_recs,
    catalog_size=catalog_size,
    topk=TOPK,
)

In [ ]:
tests.check_i2i_recs(metrics_i2i)

### 6. Item2Vec

Item-to-item can be used with any kind of item similarities.

Now, let's try and train item embeddings first, then calculate similarities and then use `item-to-item` again.

We'll be using Item2Vec approach - we'll apply **word2vec** to a sequence of items instead of sequence of words.

For that, we suggest you use `gensim`:
* you need to create `corpus` from a list of lists with strings of type `[['1', '2'], ['3', '4', '5']]`, in which strings - strings of item ids, and inner lists are grouped user interactions (sorted in chronological order)
* Call method `gensim.models.Word2Vec`. We propose `vector_size=100, window=5, min_count=1, sg=0, epochs=5`

After training, embeds can be retrieved using `w2v.wv.vectors`, and corresponding ids - using `w2v.wv.index_to_key`.

After that, use the pipeline from the previous task: `get_similar_items_gpu` + `get_candidates_item2item`

In [18]:
from gensim.models import Word2Vec

###########################
### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
###########################

train_sorted = train.sort(["uid", "timestamp"], descending=[False, False]) # ascending order to newest

user_sequence = (
    train_sorted
    .group_by("uid")
    .agg(pl.col("item_id").alias("item_seq"))
)

# build corpus
corpus = [
    [str(item_id) for item_id in seq]
    for seq in user_sequence["item_seq"].to_list()
]

In [19]:
w2v = Word2Vec(
    sentences=corpus,
    vector_size=100,
    window=5,
    min_count=1,
    sg=0,
    epochs=5,
)

In [20]:
item_ids_w2v = np.array([int(x) for x in w2v.wv.index_to_key])
item_embeddings_w2v = w2v.wv.vectors.astype(np.float32)

In [21]:
item_ids_w2v.shape, item_embeddings_w2v.shape

((157157,), (157157, 100))

In [22]:
similar_items_w2v = get_similar_items_gpu(
    item_ids=item_ids_w2v,
    item_embeddings=item_embeddings_w2v,
    block=16,
    cand_block=2048,
    topk=2 * TOPK,
    device="cuda",
)

Query blocks:   0%|          | 0/9823 [00:00<?, ?it/s]

In [23]:
w2v_recs = get_candidates_item2item(
    interactions=train,
    similar_items=similar_items_w2v,
    n_last=30,
    half_life_frac=0.5,
    topk=TOPK,
)

In [25]:
catalog_size = train.select(pl.col("item_id").n_unique()).item()

In [26]:
metrics_w2v = evaluate(
    targets=test_targets,
    candidates=w2v_recs,
    catalog_size=catalog_size,
    topk=TOPK,
)

In [27]:
metrics_w2v

{'hitrate': 0.18530684185226726,
 'recall': 0.04640406111337076,
 'ndcg': 0.017205124427572605,
 'coverage': 0.7925704868380028}

In [28]:
tests.check_w2v_recs(metrics_w2v)

### 7. Item-based Collaborative Filtering (1 балл)

Теперь попробуем применить тот же item-to-item подход, но используя в качестве векторов большие разреженные векторы из user-item матрицы. Для подсчета разреженных близостей наш GPU пайплайн не подходит (не умеет работать с разреженными данными), поэтому будем использовать библиотеку `implicit`

1. Сначала нужно собрать CSR user-item матрицу из наших `train` взаимодействий с помощью `scipy.sparse.csr_matrix`
* предлагается использовать формат вида `csr_matrix((ones, (user_ids, item_ids)), shape=(num_users, num_items))`
* чтобы просуммировать взаимодействия с одними и теми же айтемами можно использовать `.sum_duplicates`
* также понадобится от исходных айдишников (которые необязательно от 1 до `num_items / num_users`) перейти к компактным айдишникам, сделав маппинг `{old_item_id: new_item_id}`. Простой вариант -- сделать это с помощью словаря, более быстрый - использовать метод вида `train.select("uid").unique().with_row_index()` (аналогично для айтемов). И еще очень хак для ускорения - чтобы в таблице со старым `old_item_id` получить `new_item_id`, достаточно сделать джойн: `interactions.join(mapping_table, on='old_item_id', how='left')`, где название `old_item_id` зависит от имплементации (скорее всего это будет просто `item_id`)

2. Чтобы с помощью `implicit` получить списки похожих, нужно использовать комбинацию из `CosineRecommender.fit`, и `.similar_items`

3. Чтобы сформировать рекомендации, используем нашу функцию `get_candidates_item2item`

**Warning:** неаккуратно написанный код в этом пункте может переполнить оперативную память.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

from implicit.nearest_neighbours import CosineRecommender, tfidf_weight


def run_cosine(X: csr_matrix):
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################
    pass

###########################
### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
###########################

In [ ]:
tests.check_cf_recs(metrics_cf)

##### 7.2 TF-IDF

А теперь щепотка магии - с помощью `implicit.nearest_neighbours.tfidf_weight` модифицируем user-item матрицу и получим более высокие метрики.

Все, что нужно - это применить этот метод к матрице и затем проделать все те же операции, что и раньше (с вычислением похожестей и т.д.)

In [ ]:
X_tfidf = tfidf_weight(X)
metrics_cf_tfidf = run_cosine(X_tfidf)

metrics_cf_tfidf

In [ ]:
tests.check_tfidf_recs(metrics_cf_tfidf)

### 8. ALS (1 балл)

С помощью `implicit.als.AlternatingLeastSquares` над той же самой разреженной user-item матрицей можно обучить ALS, что вам и предлагается сделать.

Чтобы сформировать рекомендации, наша функция `get_candidates_item2item` не нужна - достаточно использовать метод `model.recommend`.

Нужно обучить ALS с двумя версиями user-item матриц - исходной и tfidf-модифицированной.

In [ ]:
from implicit.als import AlternatingLeastSquares

def run_als(X: csr_matrix, factors: int = 100, reg: float = 0.5, alpha: float = 0.1, iters: int = 20):
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################
    pass

metrics_als_raw = run_als(X)
metrics_als_tfidf = run_als(X_tfidf)

metrics_als_raw, metrics_als_tfidf

In [ ]:
tests.check_als_recs(metrics_als_raw, metrics_als_tfidf)

### 9. Лидерборд и выводы

Собираем таблицу со всеми методами и метриками.

Добавьте 5–10 строк выводов к экспериментам: что работает лучше и почему.


In [ ]:
leaderboard = pl.DataFrame([
    {"method": "TopPop", **metrics_toppop},
    {"method": "User-Artist", **metrics_artist},
    {"method": "Item2Item (dataset emb)", **metrics_i2i},
    {"method": "Item2Vec (w2v)", **metrics_w2v},
    {"method": "CF Cosine (raw)", **metrics_cf},
    {"method": "CF Cosine (tf-idf)", **metrics_cf_tfidf},
    {"method": "ALS (raw)", **metrics_als_raw},
    {"method": "ALS (tf-idf)", **metrics_als_tfidf},
])

leaderboard = leaderboard.sort(["recall", "ndcg"], descending=True)
leaderboard

### Вопросы на понимание (1 балл)

1) Почему для кандидатов/метрик важно исключать айтемы, которые пользователь уже видел?  
2) Почему при джойне метаданных (да и вообще почти при любом джойне) нужно использовать how='left', а не how='inner'?
3) В рекомендациях по артистам (с помощью счётчиков) не для всех пользователей может найтись нужное количество кандидатов. Почему?
4) Почему tf-idf улучшает item-based CF? На саму функцию можно посмотреть через `tfidf_weight??`
5) В чем принципиальное отличие между item-to-item методом и методом, при котором мы получаем эмбеддинг пользователя, сложив эмбеддинги его последних взаимодействий, и затем ищем ближайшие эмбеддинги айтемов?
5) Почему ALS выиграл у чистого cosine-item2item?

Ответы - текстом